# Entrega 2 — Detecção de Duplicatas

A regra de duplicata usada nesta proposta é baseada nos três pares verdadeiros registrados em `verdade_duplicatas.json`. O limiar inicial de 0,85 segue o enunciado; também testamos uma pequena faixa de thresholds para observar o compromisso entre falsos positivos e falsos negativos.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, str(Path.cwd()))
from ouvidoria import carregar, representar, detectar_duplicatas, MODELOS, cosseno_matriz

df = carregar()
verdade = {tuple(sorted(p)) for p in json.loads(Path("verdade_duplicatas.json").read_text())["pairs"]}
modelo = list(MODELOS)[0]
emb = representar(tuple(df.texto), modelo)
sim = cosseno_matriz(emb)
ids = df.id.tolist()
print("Verdade:", sorted(verdade))


In [ ]:
fig, ax = plt.subplots(figsize=(9,7))
im=ax.imshow(sim, vmin=0, vmax=1, cmap="viridis")
ax.set_title("Matriz de similaridade")
ax.set_xlabel("Manifestação")
ax.set_ylabel("Manifestação")
fig.colorbar(im, ax=ax, label="Cosseno")
fig.tight_layout()
fig.savefig("resultados/heatmap_duplicatas.png", dpi=160)
plt.show()

In [ ]:
def avaliar(threshold):
    encontrados = set()
    for i in range(len(ids)):
        for j in range(i+1,len(ids)):
            if sim[i,j] >= threshold:
                encontrados.add(tuple(sorted((ids[i],ids[j]))))
    tp = len(encontrados & verdade)
    fp = len(encontrados - verdade)
    fn = len(verdade - encontrados)
    precision = tp/(tp+fp) if tp+fp else 0
    recall = tp/(tp+fn) if tp+fn else 0
    return {"limiar":threshold,"TP":tp,"FP":fp,"FN":fn,"precisao":precision,"recall":recall,"pares":len(encontrados)}

avaliacao=pd.DataFrame([avaliar(t) for t in [0.75,0.80,0.85,0.88,0.90,0.92]])
display(avaliacao)
avaliacao.to_csv("resultados/thresholds_duplicatas.csv", index=False)


In [ ]:
pares = detectar_duplicatas(df.texto.tolist(), 0.85, modelo)
resultado = [
    {"ID 1":ids[p["i"]], "ID 2":ids[p["j"]], "cosseno":p["similaridade"]}
    for p in pares
]
display(pd.DataFrame(resultado))
Path("resultados/pares_duplicados.json").write_text(
    json.dumps(resultado, ensure_ascii=False, indent=2)
)

## Análise de erros

Falsos positivos representam manifestações semanticamente próximas que não são a mesma ocorrência; esse risco é esperado porque o embedding representa significado e não identidade. Falsos negativos podem ocorrer quando duas descrições da mesma ocorrência usam vocabulário muito diferente ou quando o texto contém contexto adicional que dilui o sinal. O limiar de 0,85 é tratado como ponto de partida, não como verdade universal; a tabela de thresholds permite justificar ajustes com base nos dados da própria atividade.